# HITL B V10: Viewport-Filling Gradio 6 Canvas Reviewer

**Runtime:** CPU.

This redesign replaces the four Gradio image panes with one browser-side canvas.

All interactive work runs locally in the browser:

- fit-to-width and fit-to-page;
- wheel zoom centred on the pointer;
- click-and-drag panning;
- confidence threshold filtering;
- confidence-weighted red heatmap;
- blue ADD brush;
- green ERASE brush;
- final-mask-only mode;
- overlay visibility controls;
- local undo and redo.

Colab is contacted only when loading a page or saving corrections. This removes the full-resolution server round trip after every brush stroke.

## Mouse controls

- **Wheel:** zoom.
- **Pan mode:** drag to pan.
- **ADD mode:** paint missing annotations.
- **ERASE mode:** paint false-positive predictions.
- **Middle mouse or Space + drag:** temporary pan.

The corrected mask is still saved as a binary PNG in `masks/corrected/`.

This version uses the supported Gradio 6 `js_on_load` lifecycle. It contains no `<script>` tag inside the HTML component.


V10 calculates the canvas height from the live browser viewport and resizes whenever the browser window changes.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q gradio pillow

## Configuration

In [ ]:
from pathlib import Path
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
SOURCE_DIR=ROOT/'source'/'annotated_only'
DRAFT_MASK_DIR=ROOT/'human_in_the_middle'/'draft_masks'
CONFIDENCE_DIR=ROOT/'human_in_the_middle'/'confidence_maps'
ALIGNED_DIR=ROOT/'aligned'
CORRECTED_DIR=ROOT/'masks'/'corrected'
AUDIT_DIR=ROOT/'human_in_the_middle'/'audit'
PREVIEW_DIR=ROOT/'human_in_the_middle'/'previews'
PUBLIC_LINK=True
DEFAULT_CONFIDENCE=0.20

for folder in [SOURCE_DIR,DRAFT_MASK_DIR,CONFIDENCE_DIR,ALIGNED_DIR,CORRECTED_DIR,AUDIT_DIR,PREVIEW_DIR]:
    folder.mkdir(parents=True,exist_ok=True)

## Backend: queue, image packaging and save handling

In [ ]:
import base64,cv2,json,hashlib,re,io,gradio as gr
import numpy as np
from PIL import Image
from datetime import datetime,timezone

SUPPORTED={'.jpg','.jpeg','.png','.tif','.tiff','.bmp'}
def files(folder):return sorted(p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED)
def md5(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1048576),b''):h.update(block)
    return h.hexdigest()
def reviewed_hashes():
    result=set()
    for path in AUDIT_DIR.glob('Page*.json'):
        try:
            value=json.loads(path.read_text()).get('source_md5')
            if value:result.add(value)
        except Exception as error:print('WARNING audit',path,error)
    return result
def next_page_number():
    values=[]
    for folder in [ALIGNED_DIR,CORRECTED_DIR]:
        for path in files(folder):
            match=re.fullmatch(r'Page(\d+)',path.stem,re.I)
            if match:values.append(int(match.group(1)))
    return max(values,default=0)+1
def build_queue():
    done=reviewed_hashes()
    return [p for p in files(SOURCE_DIR) if (DRAFT_MASK_DIR/f'{p.stem}_draft.png').exists() and md5(p) not in done]
def data_url(array,extension='.png'):
    ok,encoded=cv2.imencode(extension,array)
    if not ok:raise RuntimeError('Image encoding failed')
    mime='image/png' if extension=='.png' else 'image/jpeg'
    return f'data:{mime};base64,'+base64.b64encode(encoded.tobytes()).decode()
def confidence_to_png(confidence):
    return np.clip(confidence*255,0,255).astype(np.uint8)
def decode_mask_data_url(value,shape):
    if not value:return np.zeros(shape,np.uint8)
    payload=value.split(',',1)[1] if ',' in value else value
    raw=base64.b64decode(payload);array=np.frombuffer(raw,np.uint8);image=cv2.imdecode(array,cv2.IMREAD_GRAYSCALE)
    if image is None:raise ValueError('Correction mask could not be decoded')
    if image.shape!=shape:image=cv2.resize(image,(shape[1],shape[0]),interpolation=cv2.INTER_NEAREST)
    return np.where(image>0,255,0).astype(np.uint8)

QUEUE=build_queue();QUEUE_NAMES=[p.name for p in QUEUE]
if not QUEUE_NAMES:raise RuntimeError('No unreviewed pages with draft masks.')
QUEUE_MAP={p.name:p for p in QUEUE}

def package_page(name):
    if name not in QUEUE_MAP:raise gr.Error('Page is no longer in the active queue.')
    source=QUEUE_MAP[name];bgr=cv2.imread(str(source));draft=cv2.imread(str(DRAFT_MASK_DIR/f'{source.stem}_draft.png'),0)
    confidence_path=CONFIDENCE_DIR/f'{source.stem}_confidence.npy'
    confidence=np.load(confidence_path).astype(np.float32) if confidence_path.exists() else (draft>0).astype(np.float32)*DEFAULT_CONFIDENCE
    if draft is None or draft.shape!=bgr.shape[:2] or confidence.shape!=draft.shape:raise gr.Error('Source, draft or confidence dimensions do not match.')
    return {
        'name':name,'source_path':str(source),'source_md5':md5(source),
        'width':int(bgr.shape[1]),'height':int(bgr.shape[0]),
        'image':data_url(bgr,'.png'),'draft':data_url(draft,'.png'),
        'confidence':data_url(confidence_to_png(confidence),'.png'),
        'next_page':f'Page{next_page_number():03d}',
    }

def refresh_queue():
    global QUEUE,QUEUE_NAMES,QUEUE_MAP
    QUEUE=build_queue();QUEUE_NAMES=[p.name for p in QUEUE];QUEUE_MAP={p.name:p for p in QUEUE}

def load_backend(name):
    return json.dumps(package_page(name)),f'Loaded {name}. Next output Page{next_page_number():03d}.',gr.update(choices=QUEUE_NAMES,value=name)

def skip_backend(name):
    if name in QUEUE_NAMES and len(QUEUE_NAMES)>1:
        QUEUE_NAMES.remove(name);QUEUE_NAMES.append(name)
    next_name=QUEUE_NAMES[0]
    return json.dumps(package_page(next_name)),f'Skipped {name} for later. Loaded {next_name}.',gr.update(choices=QUEUE_NAMES,value=next_name)

def save_backend(name,add_data,erase_data,notes):
    page=package_page(name);source=Path(page['source_path']);bgr=cv2.imread(str(source));draft=cv2.imread(str(DRAFT_MASK_DIR/f'{source.stem}_draft.png'),0)
    additions=decode_mask_data_url(add_data,draft.shape);erasures=decode_mask_data_url(erase_data,draft.shape)
    final=(((draft>0)|(additions>0))&~(erasures>0)).astype(np.uint8)*255
    output_name=f'Page{next_page_number():03d}';image_output=ALIGNED_DIR/f'{output_name}.png';mask_output=CORRECTED_DIR/f'{output_name}.png'
    cv2.imwrite(str(image_output),bgr);cv2.imwrite(str(mask_output),final)
    overlay=bgr.copy();colour=overlay.copy();colour[final>0]=(0,0,255);overlay=cv2.addWeighted(overlay,.60,colour,.40,0);cv2.imwrite(str(PREVIEW_DIR/f'{output_name}_overlay.png'),overlay)
    audit={'created_utc':datetime.now(timezone.utc).isoformat(),'page':output_name,'source':str(source),'source_md5':page['source_md5'],'draft_pixels':int(np.sum(draft>0)),'added_pixels':int(np.sum(additions>0)),'erased_pixels':int(np.sum(erasures>0)),'final_pixels':int(np.sum(final>0)),'notes':notes,'aligned_output':str(image_output),'corrected_mask_output':str(mask_output)}
    (AUDIT_DIR/f'{output_name}.json').write_text(json.dumps(audit,indent=2))
    refresh_queue()
    if not QUEUE_NAMES:return '',f'Saved {output_name}. Queue complete.',gr.update(choices=[],value=None),''
    next_name=QUEUE_NAMES[0]
    return json.dumps(package_page(next_name)),f'Saved {output_name}. Loaded {next_name}.',gr.update(choices=QUEUE_NAMES,value=next_name),''

## Browser-side canvas interface

In [ ]:
CANVAS_HTML=r"""
<div id="hitl-app">
  <div id="hitl-toolbar">
    <button data-mode="pan" class="active">Pan</button>
    <button data-mode="add">ADD</button>
    <button data-mode="erase">ERASE</button>
    <button id="undo">Undo</button><button id="redo">Redo</button>
    <button id="fitWidth">Fit width</button><button id="fitPage">Fit page</button>
    <label>Brush <input id="brush" type="range" min="2" max="80" value="14"><span id="brushVal">14</span></label>
    <label>Confidence <input id="threshold" type="range" min="20" max="100" value="20"><span id="thresholdVal">0.20</span></label>
    <label>Opacity <input id="opacity" type="range" min="5" max="95" value="70"><span id="opacityVal">70%</span></label>
  </div>
  <div id="hitl-toggles">
    <label><input id="showImage" type="checkbox" checked>Original</label>
    <label><input id="showPrediction" type="checkbox" checked>Prediction</label>
    <label><input id="showAdd" type="checkbox" checked>Additions</label>
    <label><input id="showErase" type="checkbox" checked>Erasures</label>
    <label><input id="finalOnly" type="checkbox">Final mask only</label>
    <span id="zoomLabel">100%</span>
  </div>
  <div id="canvas-wrap"><canvas id="hitl-canvas"></canvas></div>
  <div id="hitl-dock">
    <button id="loadAction">Load selected</button>
    <button id="skipAction">Skip for later</button>
    <button id="saveAction" class="primary">Save + load next</button>
  </div>
</div>
"""

CANVAS_JS=r"""

 const canvas=document.getElementById('hitl-canvas'),ctx=canvas.getContext('2d',{willReadFrequently:true}),wrap=document.getElementById('canvas-wrap');
 let page=null,img=null,draft=null,conf=null,addMask=null,eraseMask=null,scale=1,ox=0,oy=0,mode='pan',drawing=false,last=null,space=false,history=[],future=[];
 const $=id=>document.getElementById(id); const imgLoad=src=>new Promise((res,rej)=>{const i=new Image();i.onload=()=>res(i);i.onerror=rej;i.src=src});
 function sizeToViewport(){const shell=document.getElementById('canvas-shell'),app=document.getElementById('hitl-app');if(!shell||!app)return;const top=shell.getBoundingClientRect().top;const reserved=92;const available=Math.max(560,window.innerHeight-top-reserved);shell.style.height=available+'px';app.style.height=available+'px';wrap.style.height=Math.max(360,available-$('hitl-toolbar').offsetHeight-$('hitl-toggles').offsetHeight)+'px';resize()}
 function maskCanvas(w,h){const c=document.createElement('canvas');c.width=w;c.height=h;return c}
 function snapshot(){history.push({a:addMask.toDataURL(),e:eraseMask.toDataURL()});if(history.length>30)history.shift();future=[]}
 async function restore(s){const [a,e]=await Promise.all([imgLoad(s.a),imgLoad(s.e)]);addMask.getContext('2d').clearRect(0,0,addMask.width,addMask.height);eraseMask.getContext('2d').clearRect(0,0,eraseMask.width,eraseMask.height);addMask.getContext('2d').drawImage(a,0,0);eraseMask.getContext('2d').drawImage(e,0,0);render()}
 function resize(){canvas.width=wrap.clientWidth*devicePixelRatio;canvas.height=wrap.clientHeight*devicePixelRatio;canvas.style.width=wrap.clientWidth+'px';canvas.style.height=wrap.clientHeight+'px';render()}
 function fitPage(){if(!page)return;scale=Math.min(wrap.clientWidth/page.width,wrap.clientHeight/page.height);ox=(wrap.clientWidth-page.width*scale)/2;oy=(wrap.clientHeight-page.height*scale)/2;render()}
 function fitWidth(){if(!page)return;scale=wrap.clientWidth/page.width;ox=0;oy=0;render()}
 function render(){ctx.setTransform(devicePixelRatio,0,0,devicePixelRatio,0,0);ctx.clearRect(0,0,wrap.clientWidth,wrap.clientHeight);if(!page)return;ctx.save();ctx.translate(ox,oy);ctx.scale(scale,scale);
   if($('showImage').checked)ctx.drawImage(img,0,0);
   const threshold=+$('threshold').value/100,opacity=+$('opacity').value/100,finalOnly=$('finalOnly').checked;
   const tmp=maskCanvas(page.width,page.height),t=tmp.getContext('2d');t.drawImage(conf,0,0);const cd=t.getImageData(0,0,page.width,page.height);const dtx=maskCanvas(page.width,page.height).getContext('2d');dtx.drawImage(draft,0,0);const dd=dtx.getImageData(0,0,page.width,page.height).data;const ax=addMask.getContext('2d').getImageData(0,0,page.width,page.height).data,ex=eraseMask.getContext('2d').getImageData(0,0,page.width,page.height).data,out=t.createImageData(page.width,page.height);
   for(let i=0;i<out.data.length;i+=4){const c=cd.data[i]/255,isDraft=dd[i]>0,isAdd=ax[i+3]>0,isErase=ex[i+3]>0,final=(isDraft||isAdd)&&!isErase;let show=false,a=0;
     if(finalOnly){show=final;a=opacity*(isAdd?1:Math.max(.08,(c-.2)/.8));}
     else if($('showPrediction').checked&&isDraft&&!isErase&&c>=threshold){show=true;a=opacity*(.08+.92*Math.max(0,(c-threshold)/(1-threshold||1)));}
     if(show){out.data[i]=255;out.data[i+1]=0;out.data[i+2]=0;out.data[i+3]=Math.round(255*a)}
   }t.putImageData(out,0,0);ctx.drawImage(tmp,0,0);
   if(!finalOnly&&$('showAdd').checked){ctx.globalAlpha=.55;ctx.drawImage(addMask,0,0)}if(!finalOnly&&$('showErase').checked){ctx.globalAlpha=.55;ctx.drawImage(eraseMask,0,0)}ctx.restore();$('zoomLabel').textContent=Math.round(scale*100)+'%'}
 function point(ev){const r=canvas.getBoundingClientRect();return {sx:ev.clientX-r.left,sy:ev.clientY-r.top,x:(ev.clientX-r.left-ox)/scale,y:(ev.clientY-r.top-oy)/scale}}
 function stroke(target,a,b,colour){const c=target.getContext('2d');c.strokeStyle=colour;c.lineWidth=+$('brush').value/scale;c.lineCap='round';c.lineJoin='round';c.beginPath();c.moveTo(a.x,a.y);c.lineTo(b.x,b.y);c.stroke()}
 canvas.onpointerdown=e=>{canvas.setPointerCapture(e.pointerId);drawing=true;last=point(e);if(mode!=='pan'&&!space)snapshot()};canvas.onpointermove=e=>{if(!drawing)return;const p=point(e);if(mode==='pan'||space||e.buttons===4){ox+=p.sx-last.sx;oy+=p.sy-last.sy}else if(mode==='add')stroke(addMask,last,p,'rgba(0,110,255,1)');else stroke(eraseMask,last,p,'rgba(0,190,90,1)');last=p;render()};canvas.onpointerup=()=>{drawing=false;last=null};
 canvas.onwheel=e=>{e.preventDefault();const p=point(e),factor=e.deltaY<0?1.15:1/1.15,newScale=Math.max(.08,Math.min(12,scale*factor));ox=p.sx-p.x*newScale;oy=p.sy-p.y*newScale;scale=newScale;render()};
 window.addEventListener('keydown',e=>{if(e.code==='Space')space=true;if((e.ctrlKey||e.metaKey)&&e.key==='z')$('undo').click()});window.addEventListener('keyup',e=>{if(e.code==='Space')space=false});window.addEventListener('resize',sizeToViewport);
 document.querySelectorAll('[data-mode]').forEach(b=>b.onclick=()=>{mode=b.dataset.mode;document.querySelectorAll('[data-mode]').forEach(x=>x.classList.toggle('active',x===b))});
 $('undo').onclick=async()=>{if(!history.length)return;future.push({a:addMask.toDataURL(),e:eraseMask.toDataURL()});await restore(history.pop())};$('redo').onclick=async()=>{if(!future.length)return;history.push({a:addMask.toDataURL(),e:eraseMask.toDataURL()});await restore(future.pop())};$('fitWidth').onclick=fitWidth;$('fitPage').onclick=fitPage;
 ['showImage','showPrediction','showAdd','showErase','finalOnly','threshold','opacity'].forEach(id=>$(id).oninput=()=>{if(id==='threshold')$('thresholdVal').textContent=(+$('threshold').value/100).toFixed(2);if(id==='opacity')$('opacityVal').textContent=$('opacity').value+'%';render()});$('brush').oninput=()=>$('brushVal').textContent=$('brush').value;
 function bridgeField(){return document.querySelector('#action-bridge textarea')||document.querySelector('#action-bridge input');}
 function notesField(){return document.querySelector('#notes-field textarea')||document.querySelector('#notes-field input');}
 function sendAction(action){const host=bridgeField();if(!host){alert('Action bridge unavailable. Reload the page.');return;}const payload={action:action,name:page?.name||'',notes:notesField()?.value||''};if(action==='save'){payload.add=addMask.toDataURL('image/png');payload.erase=eraseMask.toDataURL('image/png');}const proto=host.tagName==='TEXTAREA'?HTMLTextAreaElement.prototype:HTMLInputElement.prototype;Object.getOwnPropertyDescriptor(proto,'value').set.call(host,JSON.stringify(payload));host.dispatchEvent(new InputEvent('input',{bubbles:true,inputType:'insertText'}));host.dispatchEvent(new Event('change',{bubbles:true}));}
 $('loadAction').onclick=()=>sendAction('load');$('skipAction').onclick=()=>sendAction('skip');$('saveAction').onclick=()=>sendAction('save');
 async function ingest(raw){if(!raw)return;page=JSON.parse(raw);[img,draft,conf]=await Promise.all([imgLoad(page.image),imgLoad(page.draft),imgLoad(page.confidence)]);addMask=maskCanvas(page.width,page.height);eraseMask=maskCanvas(page.width,page.height);history=[];future=[];sizeToViewport();requestAnimationFrame(fitWidth)}
 window.hitlIngest=ingest;
 new MutationObserver(()=>{const el=document.querySelector('#page-package textarea');if(el&&el.value&&el.value!==window._lastPackage){window._lastPackage=el.value;ingest(el.value)}}).observe(document.body,{subtree:true,childList:true,characterData:true,attributes:true});
 setTimeout(()=>{sizeToViewport();const el=document.querySelector('#page-package textarea');if(el?.value)ingest(el.value)},500);setTimeout(sizeToViewport,1200);

"""

CSS=r"""
.gradio-container{max-width:100vw!important;padding:8px 8px 84px!important}
#canvas-shell{height:auto!important;min-height:0!important;padding:0!important;overflow:hidden!important}
#canvas-shell>div,#canvas-shell .html-container,#canvas-shell .prose{height:100%!important;min-height:0!important;padding:0!important;margin:0!important;overflow:hidden!important}
#hitl-app{height:100%;display:flex;flex-direction:column;background:#202124;color:#fff;font-family:Arial,sans-serif}
#hitl-toolbar,#hitl-toggles{display:flex;align-items:center;gap:8px;padding:7px 10px;background:#303134;flex-wrap:wrap}
#hitl-toolbar button,#hitl-dock button{padding:8px 13px;border:0;border-radius:7px;background:#5f6368;color:#fff;cursor:pointer}#hitl-toolbar button.active{background:#1a73e8}#hitl-dock button.primary{background:#ff6d00}
#hitl-toolbar label,#hitl-toggles label{display:flex;align-items:center;gap:4px;font-size:13px}
#canvas-wrap{flex:1;min-height:0;overflow:hidden;background:#555;cursor:crosshair}#hitl-canvas{display:block}
#hitl-dock{position:fixed;bottom:14px;left:50%;transform:translateX(-50%);z-index:9999;background:rgba(32,33,36,.96);padding:10px 14px;border-radius:15px;box-shadow:0 5px 22px rgba(0,0,0,.45);display:flex;gap:10px}
.bridge-offscreen{position:fixed!important;left:-10000px!important;top:-10000px!important;width:4px!important;height:4px!important;opacity:.01!important;pointer-events:none!important}#hitl-toolbar,#hitl-toggles,#hitl-toolbar label,#hitl-toggles label,#zoomLabel,#brushVal,#thresholdVal,#opacityVal{color:#f7f7f7!important}#hitl-toolbar button,#hitl-dock button{color:#fff!important;font-weight:700!important}footer{display:none!important}
"""

with gr.Blocks(title='HITL Single Canvas Reviewer') as app:
    with gr.Row():
        source_choice=gr.Dropdown(choices=QUEUE_NAMES,value=QUEUE_NAMES[0],label='Unreviewed source',scale=4)
        status=gr.Textbox(label='Status',interactive=False,scale=4)
        notes=gr.Textbox(label='Notes',scale=3,elem_id='notes-field')
    canvas=gr.HTML(CANVAS_HTML,elem_id='canvas-shell',js_on_load=CANVAS_JS)
    page_package=gr.Textbox(value=json.dumps(package_page(QUEUE_NAMES[0])),elem_id='page-package',elem_classes=['bridge-offscreen'])
    action_bridge=gr.Textbox(value='',elem_id='action-bridge',elem_classes=['bridge-offscreen'])
    def handle_action(command_text,selected_name):
        if not command_text:return gr.update(),gr.update(),gr.update(),gr.update()
        command=json.loads(command_text);action=command.get('action');name=command.get('name') or selected_name
        if action=='load':
            package,message,choice=load_backend(selected_name);return package,message,choice,''
        if action=='skip':
            package,message,choice=skip_backend(name);return package,message,choice,''
        if action=='save':return save_backend(name,command.get('add',''),command.get('erase',''),command.get('notes',''))
        raise gr.Error(f'Unknown action: {action}')
    action_bridge.change(handle_action,[action_bridge,source_choice],[page_package,status,source_choice,notes],show_progress='minimal')
app.launch(share=PUBLIC_LINK,debug=False,show_error=True,css=CSS)

## Notes

- The initial canvas view is fit-to-width because that best exposes small confidence dots. Click **Fit page** whenever a whole-page overview is preferred.
- The browser performs all drawing and preview composition. The public Gradio connection is used only for page changes and saves.
- Undo and redo affect ADD and ERASE layers together.
- Confidence filtering changes only the display. Saving always starts from the complete draft mask produced at confidence 0.20, then applies ADD and ERASE corrections.
- Browser code is attached through Gradio 6 `js_on_load`, the supported execution path for custom HTML components.